In [ ]:
from cache_loader import CacheLoader
from transformers import AutoTokenizer

CACHE_DIRECTORY = "cache/k=4_numlatents=32_lr=5e-3_split=50"
MODEL_ID = "Skywork/Skywork-Reward-V2-Llama-3.1-8B"
TOKENIZER = AutoTokenizer.from_pretrained(MODEL_ID)
cache_loader = CacheLoader(CACHE_DIRECTORY, TOKENIZER)

In [ ]:
from feature_labeler import FeatureLabeler
from openai import OpenAI, AsyncOpenAI

K_POSITIVE = 10
K_ZERO = 5
HELD_OUT_SET_SIZE = 50
LABELING_INSTRUCTIONS_PATH = "instructions/labeling.txt"
with open(LABELING_INSTRUCTIONS_PATH, 'r') as file:
    LABELING_INSTRUCTIONS = file.read()
CLIENT = OpenAI()
with open("instructions/prediction.txt", 'r') as file:
    PREDICTION_INSTRUCTIONS = file.read()
ASYNC_CLIENT = AsyncOpenAI()

feature_labeler = FeatureLabeler(
    cache_loader=cache_loader,
    k_positive=K_POSITIVE,
    k_zero=K_ZERO,
    held_out_set_size=HELD_OUT_SET_SIZE,
    labeling_instructions=LABELING_INSTRUCTIONS,
    client=CLIENT,
    prediction_instructions=PREDICTION_INSTRUCTIONS,
    async_client=ASYNC_CLIENT
)

In [ ]:
for i in range(32):
    await feature_labeler.process_feature(i)
    print(feature_labeler.feature_description_evaluations[i])

In [ ]:
import csv

with open('output.csv', 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['id'] + list(next(iter(feature_labeler.feature_description_evaluations.values())).keys()))
    for key, value in feature_labeler.feature_description_evaluations.items():
        writer.writerow([key] + list(value.values()))

In [ ]:
from feature_visualization import visualize_features
visualize_features(cache_loader, list(range(32)), k=100)

In [ ]:
from safetensors.torch import load_file
import torch

def print_feature_activation_frequencies(loader, feature_count):
    feature_activation_frequencies = {}
    for cache_file_feature_range in loader.cache_file_feature_ranges: 
        cache = load_file(cache_file_feature_range["path"])
        activating_features = cache["locations"].to(torch.int64)[:, 2] + cache_file_feature_range["start"]
        features, activation_frequencies = torch.unique(activating_features, return_counts=True)
        for feature, activation_frequency in zip(features.tolist(), activation_frequencies.tolist()):
            feature_activation_frequencies[feature] = feature_activation_frequencies.get(feature, 0) + activation_frequency
    for i in range(feature_count):
        print(f"Feature {i}: {feature_activation_frequencies.get(i, 0)} activations")

print_feature_activation_frequencies(loader, feature_count=32)